# 08 — Base de focos para a busca por datas (R12, tarefa A19)
O calendário é mensal e harmonizado. Para escolher **datas exatas**, o app precisa dos **focos individuais**: data, hora da passagem, satélite, posição e FRP.

**Saída:** `data/processed/focos/focos_AAAA.parquet` (um arquivo por ano, zstd, ordenado por data).

| coluna | significado |
|---|---|
| `lat`, `lon` | centro do pixel (3 casas) |
| `data`, `hora_utc` | data e hora da passagem (UTC, HHMM) |
| `satelite` | Terra, Aqua (2001–2011) ou S-NPP (2012+) |
| `frp`, `daynight` | potência radiativa (MW) e passagem diurna/noturna |
| `lat_cel`, `lon_cel` | canto SW da célula 0,25° (a mesma do `harmonized.parquet`) |
| `peso_harmonizado` | quanto este foco vale na escala VIIRS: **VIIRS = 1**; **MODIS = focos harmonizados da célula×mês ÷ nº de focos MODIS da célula×mês** |

**Por que esse peso:** no método final, `estimativa = (n_Terra + n_Aqua) / (1/k_Terra + 1/k_Aqua)`, então todo foco MODIS da mesma célula × mês vale o mesmo `1/(1/k_Terra + 1/k_Aqua)` (só Terra: `k_Terra`). Somando os pesos de um mês, sai **exatamente** o `harmonized.parquet`. Em intervalos curtos (dias), a soma é uma **aproximação**: usa o fator do mês inteiro.

**Mesmos filtros do `harmonized.parquet`:** MODIS `confidence ≥ 30`, VIIRS `n|h`, máscara de fontes estáticas, só células 0,25° do Brasil; em 2026 (API por retângulo) também a pegada de 0,1° do VIIRS no Brasil, como no notebook 06. Fontes: MODIS 2001–2011 e VIIRS S-NPP 2012–2026.

In [1]:
import duckdb, numpy as np, pandas as pd
from pathlib import Path
RAW, PROC = Path('../data/raw'), Path('../data/processed'); OUT = PROC / 'focos'; OUT.mkdir(exist_ok=True)
con = duckdb.connect()
H = (PROC / 'harmonized.parquet').as_posix(); MASK = (PROC / 'static_mask.parquet').as_posix()
viirs_arqs = sorted(p.as_posix() for p in RAW.glob('viirs-snpp_*.parquet')) + sorted(p.as_posix() for p in (RAW / 'ano_corrente').glob('viirs-snpp_*.parquet'))
modis_arqs = sorted(p.as_posix() for p in RAW.glob('modis_*.parquet') if int(p.stem.split('_')[-1]) <= 2011)
pais = [p for p in viirs_arqs if 'ano_corrente' not in p]
con.execute(f"CREATE OR REPLACE TABLE pegada AS SELECT DISTINCT floor(latitude/0.1)::INT by_, floor(longitude/0.1)::INT bx FROM read_parquet({pais})")
con.execute(f'''
CREATE OR REPLACE TABLE pts AS
SELECT * FROM (
  SELECT latitude, longitude, CAST(acq_date AS DATE) data, CAST(acq_time AS SMALLINT) hora_utc, satellite AS satelite, frp, daynight
    FROM read_parquet({modis_arqs}) WHERE TRY_CAST(confidence AS INT) >= 30
  UNION ALL
  SELECT latitude, longitude, CAST(acq_date AS DATE), CAST(acq_time AS SMALLINT), 'S-NPP', frp, daynight
    FROM read_parquet({viirs_arqs}, union_by_name = true) v
    WHERE confidence IN ('n', 'h')
      -- 2026 vem da API por retângulo: mesmo recorte do notebook 06 (pegada de 0,1° do VIIRS no Brasil, 2012–2025)
      AND (year(CAST(acq_date AS DATE)) < 2026 OR (floor(latitude/0.1)::INT, floor(longitude/0.1)::INT) IN (SELECT (by_, bx) FROM pegada))) p
ANTI JOIN '{MASK}' m ON floor(p.latitude/0.02)::INT = m.fy AND floor(p.longitude/0.02)::INT = m.fx''')
con.execute('''CREATE OR REPLACE TABLE pts2 AS SELECT *, floor(latitude/0.25)*0.25 lat_cel, floor(longitude/0.25)*0.25 lon_cel,
                 year(data) ano, month(data) mes FROM pts''')
# só células do Brasil (as do harmonized) e o peso: harmonizado da célula×mês ÷ nº de pontos da célula×mês
con.execute(f'''
CREATE OR REPLACE TABLE focos AS
WITH n AS (SELECT lat_cel, lon_cel, ano, mes, count(*) n FROM pts2 GROUP BY ALL)
SELECT round(p.latitude, 3)::FLOAT lat, round(p.longitude, 3)::FLOAT lon, p.data, p.hora_utc, p.satelite,
       round(p.frp, 1)::FLOAT frp, p.daynight, p.lat_cel::FLOAT lat_cel, p.lon_cel::FLOAT lon_cel, p.ano,
       CASE WHEN p.satelite = 'S-NPP' THEN 1.0 ELSE h.focos_harmonizados / n.n END AS peso_harmonizado
FROM pts2 p JOIN n USING (lat_cel, lon_cel, ano, mes)
JOIN '{H}' h ON h.lat = p.lat_cel AND h.lon = p.lon_cel AND h.ano = p.ano AND h.mes = p.mes''')
print(con.execute('SELECT satelite, count(*) focos, min(data), max(data) FROM focos GROUP BY 1 ORDER BY 1').df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  satelite     focos min("data") max("data")
0     Aqua   3020851  2002-07-04  2011-12-31
1    S-NPP  17725306  2012-01-20  2026-11-02
2    Terra   1722492  2001-01-01  2011-12-31


## Teste obrigatório: Σ peso por célula × mês = `harmonized.parquet` (tolerância 1e-6)

In [2]:
t = con.execute(f'''
WITH s AS (SELECT lat_cel, lon_cel, ano, month(data) mes, sum(peso_harmonizado) soma FROM focos GROUP BY ALL)
SELECT count(*) linhas_h, count(s.soma) com_focos, max(abs(coalesce(s.soma, 0) - h.focos_harmonizados)) dif_max
FROM '{H}' h LEFT JOIN s ON s.lat_cel = h.lat AND s.lon_cel = h.lon AND s.ano = h.ano AND s.mes = h.mes''').df()
display(t)
assert t.linhas_h[0] == t.com_focos[0], 'há célula×mês do harmonized sem focos na base'
assert t.dif_max[0] < 1e-6, 'soma dos pesos diferente do harmonized'
print('OK: a soma dos pesos reproduz o harmonized.parquet em todas as células × mês')

,linhas_h,com_focos,dif_max
0,1252185,1252185,1.073204e-10


OK: a soma dos pesos reproduz o harmonized.parquet em todas as células × mês


## Gravar um arquivo por ano

In [3]:
for f in OUT.glob('focos_*.parquet'): f.unlink()
anos = [r[0] for r in con.execute('SELECT DISTINCT ano FROM focos ORDER BY 1').fetchall()]
for a in anos:
    con.execute(f'''COPY (SELECT * EXCLUDE (ano) FROM focos WHERE ano = {a} ORDER BY data, hora_utc)
                    TO '{(OUT / f"focos_{a}.parquet").as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 250000)''')
tam = pd.DataFrame([(f.name, round(f.stat().st_size / 1e6, 1)) for f in sorted(OUT.glob('focos_*.parquet'))], columns=['arquivo', 'MB'])
print(f'{len(tam)} arquivos, total {tam.MB.sum():.0f} MB, maior {tam.MB.max()} MB'); assert tam.MB.max() < 50
tam.T

26 arquivos, total 184 MB, maior 12.9 MB


,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
arquivo,focos_2001.parquet,focos_2002.parquet,focos_2003.parquet,focos_2004.parquet,focos_2005.parquet,focos_2006.parquet,focos_2007.parquet,focos_2008.parquet,focos_2009.parquet,focos_2010.parquet,...,focos_2017.parquet,focos_2018.parquet,focos_2019.parquet,focos_2020.parquet,focos_2021.parquet,focos_2022.parquet,focos_2023.parquet,focos_2024.parquet,focos_2025.parquet,focos_2026.parquet
MB,1.4,5.0,5.2,6.0,6.0,4.0,6.7,3.5,2.4,5.5,...,10.5,6.3,9.9,11.3,9.2,9.6,9.2,12.9,5.9,4.7


## Exemplo de uso (o que o app faz): Pantanal, 10–25 de agosto, todos os anos

In [4]:
import time; t0 = time.perf_counter()
ex = con.execute(f'''SELECT year(data) ano, count(*) focos_brutos, round(sum(peso_harmonizado)) focos_harmonizados_aprox,
                            any_value(satelite) sensor
   FROM read_parquet('{(OUT).as_posix()}/focos_*.parquet')
   WHERE lat BETWEEN -20 AND -16 AND lon BETWEEN -58 AND -55 AND month(data) = 8 AND day(data) BETWEEN 10 AND 25
   GROUP BY 1 ORDER BY 1''').df()
print(f'consulta em {time.perf_counter() - t0:.2f} s'); ex

consulta em 0.25 s


,ano,focos_brutos,focos_harmonizados_aprox,sensor
0,2001,1461,13965.0,Terra
1,2002,2810,9983.0,Terra
2,2003,430,1590.0,Terra
3,2004,1075,3960.0,Terra
4,2005,4223,15740.0,Aqua
5,2006,777,2828.0,Aqua
6,2007,1530,5673.0,Terra
7,2008,455,1705.0,Terra
8,2009,553,1984.0,Terra
9,2010,973,3672.0,Aqua


**Leitura:** a coluna `focos_brutos` muda de escala em 2012 (troca de sensor). A coluna `focos_harmonizados_aprox` usa o peso e fica comparável entre anos; antes de 2012 é aproximada (fator do mês aplicado a 16 dias) e deve vir com a faixa de incerteza.